# Einops for Deep Learning: Exercises

This notebook teaches einops for deep learning through **worked examples followed by exercises** (deliberate practice).

Each section introduces a pattern with a worked example explaining the *why*, then gives you exercises to drill the pattern until it's automatic.

## Setup

In [ ]:
import torch
import numpy as np
from einops import rearrange, reduce, repeat, asnumpy
from einops.layers.torch import Rearrange, Reduce

In [ ]:
# Dummy batch of images: batch=2, channels=3, height=64, width=64
x = torch.randn(2, 3, 64, 64)
print(f"x.shape = {x.shape}")

---
## Section 1: Format Conversion (BCHW <-> BHWC)

PyTorch uses **BCHW** (batch, channels, height, width) while TensorFlow/JAX often use **BHWC** (batch, height, width, channels). When porting models or using libraries that expect a different format, you need to convert between them.

With raw PyTorch you'd write `x.permute(0, 2, 3, 1)` — but the einops version is self-documenting.

### Worked Example: BCHW -> BHWC

In [ ]:
# Convert from PyTorch format (BCHW) to TensorFlow format (BHWC)
y = rearrange(x, 'b c h w -> b h w c')
print(f"Input:  {x.shape}")   # (2, 3, 64, 64)
print(f"Output: {y.shape}")   # (2, 64, 64, 3)

### Exercise 1.1

Convert a BHWC tensor back to BCHW format.

In [ ]:
bhwc_tensor = torch.randn(2, 64, 64, 3)

# YOUR CODE HERE
result = ...

assert result.shape == (2, 3, 64, 64), f"Expected (2, 3, 64, 64), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(bhwc_tensor, 'b h w c -> b c h w')
```
</details>

### Exercise 1.2

Given a **single image** (no batch dimension) of shape `(3, 64, 64)`, convert to `(64, 64, 3)` — e.g., for matplotlib which expects HWC.

In [ ]:
single_image = torch.randn(3, 64, 64)

# YOUR CODE HERE
result = ...

assert result.shape == (64, 64, 3), f"Expected (64, 64, 3), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(single_image, 'c h w -> h w c')
```
</details>

---
## Section 2: Flattening for Linear Layers

Before passing a feature map into a fully connected layer, you need to flatten the spatial and channel dimensions into a single vector. In raw PyTorch: `x.reshape(x.shape[0], -1)`. With einops, the pattern is explicit about *which* dimensions get merged.

### Worked Example: Flatten all non-batch dims

In [ ]:
# Flatten channels, height, width into one dimension for a linear layer
y = rearrange(x, 'b c h w -> b (c h w)')
print(f"Input:  {x.shape}")   # (2, 3, 64, 64)
print(f"Output: {y.shape}")   # (2, 12288)  since 3*64*64 = 12288

### Exercise 2.1

Flatten **only** the spatial dimensions (h, w) while keeping batch and channel separate: `(b, c, h, w) -> (b, c, h*w)`. This is useful for applying attention over spatial positions per channel.

In [ ]:
# YOUR CODE HERE
result = ...

assert result.shape == (2, 3, 4096), f"Expected (2, 3, 4096), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(x, 'b c h w -> b c (h w)')
```
</details>

### Exercise 2.2

**Predict** the resulting shape of `rearrange(x, 'b c h w -> b (c h w)')` when `x` has shape `(2, 3, 64, 64)`. Write your prediction, then verify.

In [ ]:
# Write your prediction here:
predicted_shape = ...  # YOUR CODE HERE: e.g., (2, ???)

# Verify
actual = rearrange(x, 'b c h w -> b (c h w)')
print(f"Actual shape: {actual.shape}")
assert predicted_shape == actual.shape, f"Your prediction {predicted_shape} didn't match {actual.shape}"
print("Correct prediction!")

<details><summary>Solution</summary>

```python
predicted_shape = (2, 12288)  # 3 * 64 * 64 = 12288
```
</details>

---
## Section 3: Space-to-Depth and Depth-to-Space

**Space-to-depth** trades spatial resolution for more channels. It takes local patches (e.g., 2x2) and stacks them along the channel dimension. This is used in efficient architectures (e.g., as a replacement for strided convolutions) and in sub-pixel convolution layers (depth-to-space is the reverse, used for super-resolution).

The key insight: `(h h1)` decomposes `h` into `(h_out, patch_h)`. So `h1=2` means every pair of rows becomes part of a 2-high patch.

### Worked Example: Space-to-Depth (2x2)

In [ ]:
# Space-to-depth: halve spatial resolution, 4x the channels
# (h h1) decomposes h=64 into (32, 2), similarly for w
# Then h1, w1 patch pixels move into the channel dim
y = rearrange(x, 'b c (h h1) (w w1) -> b (h1 w1 c) h w', h1=2, w1=2)
print(f"Input:  {x.shape}")   # (2, 3, 64, 64)
print(f"Output: {y.shape}")   # (2, 12, 32, 32)  since 2*2*3 = 12 channels

### Exercise 3.1

Implement **depth-to-space** (the reverse of space-to-depth). Given a tensor of shape `(2, 12, 32, 32)`, produce shape `(2, 3, 64, 64)` with `h1=2, w1=2`.

This is used in **sub-pixel convolution** for super-resolution: the network outputs many channels, then depth-to-space rearranges them into higher-resolution spatial pixels.

In [ ]:
depth_tensor = torch.randn(2, 12, 32, 32)

# YOUR CODE HERE
result = ...

assert result.shape == (2, 3, 64, 64), f"Expected (2, 3, 64, 64), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(depth_tensor, 'b (h1 w1 c) h w -> b c (h h1) (w w1)', h1=2, w1=2)
```
</details>

### Exercise 3.2

Implement space-to-depth with **4x4 patches** instead of 2x2. What is the output shape? Predict first, then verify.

In [ ]:
# YOUR CODE HERE
result = ...

print(f"Output shape: {result.shape}")
# Verify your mental math:
# channels should be 4*4*3 = 48, spatial should be 64/4 = 16
assert result.shape == (2, 48, 16, 16), f"Expected (2, 48, 16, 16), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(x, 'b c (h h1) (w w1) -> b (h1 w1 c) h w', h1=4, w1=4)
```

Output shape: `(2, 48, 16, 16)` — 4x4=16 patch pixels times 3 channels = 48. Spatial dims shrink from 64 to 16 (divided by 4).
</details>

---
## Section 4: Pooling Operations

Einops `reduce` lets you express pooling operations with explicit patterns. Instead of memorizing `nn.MaxPool2d(kernel_size=2, stride=2)`, you write a pattern that shows exactly which dimensions get collapsed and how.

### Worked Example: Global Average Pooling

In [ ]:
# Global average pooling: collapse all spatial dims, keep batch and channel
# This is what you'd use right before the final classifier in a CNN
y = reduce(x, 'b c h w -> b c', 'mean')
print(f"Input:  {x.shape}")   # (2, 3, 64, 64)
print(f"Output: {y.shape}")   # (2, 3)

### Exercise 4.1

Implement **2x2 max pooling**: reduce spatial resolution by 2 in each dimension using max. This is the standard pooling layer in CNNs like VGG.

Hint: decompose `h` into `(h_out, 2)` and reduce over the `2`.

In [ ]:
# YOUR CODE HERE
result = ...

assert result.shape == (2, 3, 32, 32), f"Expected (2, 3, 32, 32), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = reduce(x, 'b c (h h1) (w w1) -> b c h w', 'max', h1=2, w1=2)
```
</details>

### Exercise 4.2

Implement **1D temporal max pooling** with pool size 2 on a sequence tensor. This is used in temporal CNNs for audio or time series.

In [ ]:
seq = torch.randn(4, 100, 64)  # (batch, time, channels)

# YOUR CODE HERE
result = ...

assert result.shape == (4, 50, 64), f"Expected (4, 50, 64), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = reduce(seq, 'b (t t1) c -> b t c', 'max', t1=2)
```
</details>

### Exercise 4.3

Implement **3D average pooling** with a 2x2x2 kernel on a 3D feature map. This is used in video models and 3D medical imaging.

In [ ]:
vol = torch.randn(2, 3, 16, 16, 16)  # (batch, channels, depth, height, width)

# YOUR CODE HERE
result = ...

assert result.shape == (2, 3, 8, 8, 8), f"Expected (2, 3, 8, 8, 8), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = reduce(vol, 'b c (d d1) (h h1) (w w1) -> b c d h w', 'mean', d1=2, h1=2, w1=2)
```
</details>

---
## Section 5: Squeeze and Unsqueeze

Adding or removing size-1 dimensions is extremely common. You unsqueeze to add a batch dimension before inference, then squeeze to remove it after. In raw PyTorch: `x.unsqueeze(0)` / `x.squeeze(0)`. Einops makes the intent explicit by naming every dimension.

### Worked Example: Add batch dimension

In [ ]:
# Single image -> batched image for model inference
image = torch.randn(3, 64, 64)
batched = rearrange(image, 'c h w -> 1 c h w')
print(f"Input:  {image.shape}")    # (3, 64, 64)
print(f"Output: {batched.shape}")  # (1, 3, 64, 64)

### Exercise 5.1

After inference, your model outputs class predictions with shape `(1, 10)`. Remove the batch dimension to get shape `(10,)` — e.g., for argmax to get the predicted class.

In [ ]:
predictions = torch.randn(1, 10)

# YOUR CODE HERE
result = ...

assert result.shape == (10,), f"Expected (10,), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(predictions, '1 classes -> classes')
```
</details>

### Exercise 5.2

Add **both** a batch dimension and a time dimension to a single frame: `(c, h, w) -> (1, 1, c, h, w)`. This is useful when feeding a single frame into a video model that expects `(batch, time, channels, height, width)`.

In [ ]:
frame = torch.randn(3, 64, 64)

# YOUR CODE HERE
result = ...

assert result.shape == (1, 1, 3, 64, 64), f"Expected (1, 1, 3, 64, 64), got {result.shape}"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(frame, 'c h w -> 1 1 c h w')
```
</details>

---
## Section 6: Normalization with Broadcasting

A powerful pattern: use `reduce` to compute statistics, keeping size-1 dims so the result broadcasts back against the original tensor. The `1` in the output pattern creates a dimension of size 1, which PyTorch will broadcast.

This lets you implement different normalization styles just by changing which axes you reduce over.

### Worked Example: Instance Normalization (mean subtraction)

In [ ]:
# Per-channel, per-image mean subtraction (instance norm style)
# Reduce over spatial dims (h, w), keep batch and channel with size-1 spatial dims
mean = reduce(x, 'b c h w -> b c 1 1', 'mean')
print(f"Mean shape: {mean.shape}")  # (2, 3, 1, 1) — broadcasts against (2, 3, 64, 64)
y = x - mean
print(f"Normalized shape: {y.shape}")  # (2, 3, 64, 64)

# Verify: per-channel, per-image mean should now be ~0
new_mean = reduce(y, 'b c h w -> b c', 'mean')
print(f"Per-channel means after normalization (should be ~0):\n{new_mean}")

### Exercise 6.1

Implement **batch-norm style** mean subtraction: compute the mean over the batch AND spatial dimensions, keeping only the channel dimension. The result should broadcast as `(1, c, 1, 1)` against `(b, c, h, w)`.

In batch norm, statistics are shared across the batch and spatial positions but separate per channel.

In [ ]:
# YOUR CODE HERE
bn_mean = ...

assert bn_mean.shape == (1, 3, 1, 1), f"Expected (1, 3, 1, 1), got {bn_mean.shape}"
y = x - bn_mean
print(f"BN mean shape: {bn_mean.shape}")
print("Passed!")

<details><summary>Solution</summary>

```python
bn_mean = reduce(x, 'b c h w -> 1 c 1 1', 'mean')
```
</details>

### Exercise 6.2

Implement **layer-norm style** mean subtraction: compute the mean over channels AND spatial dimensions, keeping only the batch dimension. Each image in the batch gets its own mean.

In layer norm, statistics are computed per-sample across all features.

In [ ]:
# YOUR CODE HERE
ln_mean = ...

assert ln_mean.shape == (2, 1, 1, 1), f"Expected (2, 1, 1, 1), got {ln_mean.shape}"
y = x - ln_mean
print(f"LN mean shape: {ln_mean.shape}")
print("Passed!")

<details><summary>Solution</summary>

```python
ln_mean = reduce(x, 'b c h w -> b 1 1 1', 'mean')
```
</details>

---
## Section 7: Channel Shuffle (ShuffleNet)

In **group convolutions**, channels are split into groups and each group is convolved independently. The problem: groups never exchange information. **Channel shuffle** fixes this by interleaving channels from different groups.

The pattern: decompose channels into `(groups, channels_per_group)`, then swap the order to `(channels_per_group, groups)`. When flattened back, channels from different groups are now interleaved.

### Worked Example: Channel Shuffle

In [ ]:
# Create tensor with 12 channels (divisible by 4 groups)
shuffle_x = torch.randn(2, 12, 8, 8)

# Channel shuffle: decompose 12 channels into 4 groups of 3
# Then reorder so channels from different groups are interleaved
y = rearrange(shuffle_x, 'b (g c) h w -> b (c g) h w', g=4)
print(f"Input:  {shuffle_x.shape}")  # (2, 12, 8, 8)
print(f"Output: {y.shape}")          # (2, 12, 8, 8) — same shape, different order!

# To see the effect: channels [0,1,2] were group 0, [3,4,5] were group 1, etc.
# After shuffle: channel 0 is from group 0, channel 1 from group 1, etc.

### Exercise 7.1

Given a tensor with **16 channels**, perform channel shuffle with **8 groups** (2 channels per group).

In [ ]:
x16 = torch.randn(2, 16, 8, 8)

# YOUR CODE HERE
result = ...

assert result.shape == (2, 16, 8, 8), f"Expected (2, 16, 8, 8), got {result.shape}"
# Verify it actually shuffled (not identity)
assert not torch.equal(result, x16), "Result should be different from input (channels should be reordered)"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(x16, 'b (g c) h w -> b (c g) h w', g=8)
```
</details>

### Exercise 7.2

Implement a **two-level shuffle**: decompose channels into `(g1, g2, c)` and swap `g1` and `g2`. This creates a more complex interleaving pattern.

Use `g1=2, g2=4` on a tensor with 32 channels.

In [ ]:
x32 = torch.randn(2, 32, 8, 8)

# YOUR CODE HERE
result = ...

assert result.shape == (2, 32, 8, 8), f"Expected (2, 32, 8, 8), got {result.shape}"
assert not torch.equal(result, x32), "Result should be different from input"
print("Passed!")

<details><summary>Solution</summary>

```python
result = rearrange(x32, 'b (g1 g2 c) h w -> b (g2 g1 c) h w', g1=2, g2=4)
```
</details>

---
## Section 8: Einops Layers in nn.Sequential

Einops provides `Rearrange` and `Reduce` as `nn.Module` layers, so you can use them directly in `nn.Sequential`. This eliminates the need for custom `Flatten` or `Permute` modules and makes the architecture self-documenting.

### Worked Example: CNN with einops layers

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 16, 3, padding=1),
    Reduce('b c (h 2) (w 2) -> b c h w', 'max'),    # 2x2 max pool
    torch.nn.Conv2d(16, 32, 3, padding=1),
    Reduce('b c (h 2) (w 2) -> b c h w', 'max'),    # 2x2 max pool
    Rearrange('b c h w -> b (c h w)'),                # flatten
    torch.nn.Linear(32 * 16 * 16, 10),
)

# Test it
out = model(x)
print(f"Input:  {x.shape}")    # (2, 3, 64, 64)
print(f"Output: {out.shape}")  # (2, 10)

### Exercise 8.1

Modify the model above to:
1. Use **average pooling** instead of max pooling
2. Use **global average pooling** before the linear layer (so the linear input is just 32, not 32*16*16)

This is the modern approach (used in ResNet, EfficientNet, etc.) — global average pooling makes the model resolution-independent.

In [ ]:
# YOUR CODE HERE
model_gap = torch.nn.Sequential(
    # ...
)

out = model_gap(x)
assert out.shape == (2, 10), f"Expected (2, 10), got {out.shape}"
print(f"Output: {out.shape}")
print("Passed!")

<details><summary>Solution</summary>

```python
model_gap = torch.nn.Sequential(
    torch.nn.Conv2d(3, 16, 3, padding=1),
    Reduce('b c (h 2) (w 2) -> b c h w', 'mean'),    # 2x2 avg pool
    torch.nn.Conv2d(16, 32, 3, padding=1),
    Reduce('b c (h 2) (w 2) -> b c h w', 'mean'),    # 2x2 avg pool
    Reduce('b c h w -> b c', 'mean'),                  # global avg pool
    torch.nn.Linear(32, 10),
)
```
</details>

### Exercise 8.2

Build a network that:
1. Takes CIFAR-sized input: `(b, 3, 32, 32)`
2. Uses **space-to-depth** (2x2) as the first layer — this replaces the usual first strided conv and gives `(b, 12, 16, 16)`
3. A `Conv2d` layer (12 input channels, 32 output channels, kernel 3, padding 1)
4. **Global average pooling** to `(b, 32)`
5. A `Linear` layer to 10 classes

Space-to-depth as a stem is used in some efficient architectures — it preserves all information (unlike strided conv or pooling) while reducing spatial resolution.

In [ ]:
cifar_input = torch.randn(4, 3, 32, 32)

# YOUR CODE HERE
model_s2d = torch.nn.Sequential(
    # ...
)

out = model_s2d(cifar_input)
assert out.shape == (4, 10), f"Expected (4, 10), got {out.shape}"
print(f"Output: {out.shape}")
print("Passed!")

<details><summary>Solution</summary>

```python
model_s2d = torch.nn.Sequential(
    Rearrange('b c (h h1) (w w1) -> b (h1 w1 c) h w', h1=2, w1=2),  # space-to-depth
    torch.nn.Conv2d(12, 32, 3, padding=1),
    Reduce('b c h w -> b c', 'mean'),   # global avg pool
    torch.nn.Linear(32, 10),
)
```
</details>